# Framework Integrations — Medha with LangChain, LlamaIndex, and Haystack

This notebook shows how to plug **Medha** as a semantic cache into three popular
AI framework stacks. Each integration uses the framework's native extension point
so the cache is transparent to the rest of the pipeline.

| Framework | Integration point | Medha role |
|---|---|---|
| **LangChain** | `langchain_core.caches.BaseCache` | Intercepts LLM calls via `set_llm_cache()` |
| **LlamaIndex** | Custom `QueryPipelineComponent` | Pre-filter before the query engine |
| **Haystack** | `@component` in a `Pipeline` | First stage, short-circuits if cache hit |

All examples use `InMemoryBackend` — **no Qdrant, no PostgreSQL, no API keys required**.
A tiny mock SQL generator simulates the LLM so the notebook is fully self-contained.

**Requirements:**
```bash
pip install "medha-archai[fastembed]"
pip install langchain-core langchain-community
pip install llama-index-core
pip install haystack-ai
```

## Cell 1: Shared Setup

A single `Medha` instance and a mock SQL generator used by all three frameworks.

In [ ]:
import asyncio
import time
from medha import Medha, InMemoryBackend, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

# --- Shared Medha instance (InMemoryBackend — zero infra) ---
embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")
settings = Settings(backend_type="memory", score_threshold_semantic=0.82)
medha = Medha("framework_demo", embedder=embedder, settings=settings)
await medha.start()

# --- Mock LLM: simulates a Text-to-SQL model (no API key needed) ---
_SQL_MAP = {
    "users":     "SELECT COUNT(*) FROM users",
    "revenue":   "SELECT SUM(amount) FROM invoices WHERE YEAR(created_at) = YEAR(NOW())",
    "products":  "SELECT * FROM products ORDER BY price DESC LIMIT 10",
    "orders":    "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()",
    "salary":    "SELECT AVG(salary) FROM employees",
}

_llm_call_count = 0

def mock_llm(question: str) -> str:
    """Simulated LLM: returns a SQL query for the question."""
    global _llm_call_count
    _llm_call_count += 1
    q = question.lower()
    for keyword, sql in _SQL_MAP.items():
        if keyword in q:
            return sql
    return f"SELECT * FROM unknown  -- question: {question}"

# --- Warm the cache with a few known pairs ---
_seed_pairs = [
    ("How many users are registered?",         "SELECT COUNT(*) FROM users"),
    ("What is the total revenue this year?",   "SELECT SUM(amount) FROM invoices WHERE YEAR(created_at) = YEAR(NOW())"),
    ("Show the top 10 most expensive products","SELECT * FROM products ORDER BY price DESC LIMIT 10"),
    ("Count orders placed today",              "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
    ("What is the average employee salary?",   "SELECT AVG(salary) FROM employees"),
]
for q, sql in _seed_pairs:
    await medha.store(q, sql)

print(f"Medha warmed with {len(_seed_pairs)} entries.")
print(f"Backend: {type(medha._backend).__name__}")

---
## Part 1 — LangChain

LangChain exposes `langchain_core.caches.BaseCache` as the standard cache interface.
Any object implementing `lookup()` / `update()` / `clear()` can be registered globally
with `set_llm_cache()` — every subsequent LLM call checks the cache first.

### How it works

```
chain.invoke(question)
    └─ LLM.__call__(prompt)
           └─ cache.lookup(prompt, llm_string)   ← Medha.search()
                  hit  → return cached Generation
                  miss → call LLM → cache.update() → return
```

The prompt passed to `lookup()` is the **full rendered prompt**, not just the question.
For Text-to-Query we extract the user question from the prompt before calling Medha.

In [ ]:
from langchain_core.caches import BaseCache
from langchain_core.outputs import Generation
from langchain_core.globals import set_llm_cache
from typing import Optional
import concurrent.futures


def _run_async(coro):
    """Run a coroutine from sync context without conflicting with a running event loop."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


class MedhaLangChainCache(BaseCache):
    """Wraps a Medha instance as a LangChain LLM cache."""

    def __init__(self, medha_instance: Medha) -> None:
        self._medha = medha_instance

    def lookup(self, prompt: str, llm_string: str) -> Optional[list[Generation]]:
        hit = _run_async(self._medha.search(prompt))
        if hit.strategy == SearchStrategy.NO_MATCH:
            return None
        return [Generation(text=hit.generated_query)]

    def update(self, prompt: str, llm_string: str, return_val: list[Generation]) -> None:
        sql = return_val[0].text if return_val else ""
        _run_async(self._medha.store(prompt, sql))

    def clear(self, **kwargs) -> None:
        pass


set_llm_cache(MedhaLangChainCache(medha))
print("MedhaLangChainCache registered as global LangChain cache.")

In [ ]:
_lc_llm_calls = 0

async def langchain_text2sql(question: str) -> tuple[str, str]:
    """Returns (sql, resolver) where resolver is 'cache' or 'llm'."""
    global _lc_llm_calls
    cached = await medha.search(question)
    if cached.strategy != SearchStrategy.NO_MATCH:
        return cached.generated_query, "cache"
    _lc_llm_calls += 1
    sql = mock_llm(question)
    await medha.store(question, sql)
    return sql, "llm"

test_questions_lc = [
    ("Show the top 10 most expensive products",  "seed — cache hit"),
    ("Top products ranked by cost",               "semantic hit"),
    ("Count orders placed today",                 "seed — cache hit"),
    ("How many orders were submitted today?",     "semantic hit"),
    ("List all tables in the database",           "miss — LLM called"),
]

print("LangChain integration — results:")
print(f"  {'Question':<45s}  {'Resolved by':<12s}  Note")
print("  " + "-" * 85)
for question, note in test_questions_lc:
    _, resolver = await langchain_text2sql(question)
    print(f"  {question:<45s}  {resolver:<12s}  {note}")

print(f"\nLLM calls made: {_lc_llm_calls} / {len(test_questions_lc)} requests")

---
## Part 2 — LlamaIndex

LlamaIndex 0.10.38+ introduced **Workflows** as the recommended way to build
multi-step pipelines. A Workflow is made of typed `Event` objects and `@step`
decorated async methods.

```
StartEvent(question)
    └─ check_cache  ──hit──→ StopEvent(sql)
                    ──miss─→ CacheMissEvent(question)
                                  └─ call_llm → store → StopEvent(sql)
```

- **`check_cache`** queries Medha; short-circuits on hit.
- **`call_llm`** only fires on `CacheMissEvent`; stores the result back in Medha.

In [ ]:
from llama_index.core.workflow import (
    Workflow,
    step,
    Event,
    StartEvent,
    StopEvent,
    Context,
)
from typing import Union


class CacheMissEvent(Event):
    question: str


class MedhaWorkflow(Workflow):
    """LlamaIndex Workflow: cache-first Text-to-SQL with Medha."""

    def __init__(self, medha_instance: Medha, **kwargs) -> None:
        super().__init__(**kwargs)
        self._medha = medha_instance
        self.llm_call_count: int = 0
        self._last_strategy: str = "no_match"

    @step
    async def check_cache(
        self, ctx: Context, ev: StartEvent
    ) -> Union[CacheMissEvent, StopEvent]:
        question: str = ev.question
        hit = await self._medha.search(question)
        self._last_strategy = hit.strategy.value
        if hit.strategy != SearchStrategy.NO_MATCH:
            return StopEvent(result=hit.generated_query)
        return CacheMissEvent(question=question)

    @step
    async def call_llm(self, ctx: Context, ev: CacheMissEvent) -> StopEvent:
        self.llm_call_count += 1
        sql = mock_llm(ev.question)
        await self._medha.store(ev.question, sql)
        return StopEvent(result=sql)


workflow = MedhaWorkflow(medha, timeout=30, verbose=False)
print("LlamaIndex Workflow built: check_cache → (miss) → call_llm")

In [ ]:
test_questions_li = [
    ("Show the top 10 most expensive products",  "seed — cache hit"),
    ("Top products ranked by cost",               "semantic hit"),
    ("Count orders placed today",                 "seed — cache hit"),
    ("How many orders were submitted today?",     "semantic hit"),
    ("List all tables in the database",           "miss — LLM called"),
]

print("LlamaIndex Workflow — results:")
print(f"  {'Question':<45s}  {'Resolved by':<12s}  Note")
print("  " + "-" * 85)

for question, note in test_questions_li:
    await workflow.run(question=question)
    resolver = "cache" if workflow._last_strategy != "no_match" else "llm"
    print(f"  {question:<45s}  {resolver:<12s}  {note}")

print(f"\nLLM calls made: {workflow.llm_call_count} / {len(test_questions_li)} requests")

---
## Part 3 — Haystack

Haystack 2.x uses a declarative `@component` model. Each component declares
its input/output types and implements a `run()` method.

We create a `MedhaCacheComponent` that returns:
- `{"sql": <cached_sql>}` on hit — the pipeline can route directly to output
- `{"question": <question>}` on miss — routed to the LLM component

```
Pipeline:
  question → MedhaCacheComponent
                  ├─ hit  → output (sql)
                  └─ miss → MockLLMComponent → MedhaStoreComponent → output (sql)
```

In [ ]:
from haystack import component, Pipeline
from typing import Optional
import concurrent.futures


def _run_async(coro):
    """Run an async coroutine from a sync context (safe inside a running event loop)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


@component
class HaystackMedhaCacheComponent:
    """Returns cached SQL (if found) and always forwards the question."""

    def __init__(self, medha_instance: Medha) -> None:
        self._medha = medha_instance
        self.hit_count = 0
        self.miss_count = 0

    @component.output_types(sql=Optional[str], question=str)
    def run(self, question: str) -> dict:
        hit = _run_async(self._medha.search(question))
        if hit.strategy != SearchStrategy.NO_MATCH:
            self.hit_count += 1
            return {"sql": hit.generated_query, "question": question}
        self.miss_count += 1
        return {"sql": None, "question": question}


@component
class HaystackMockLLMComponent:
    """Returns cached SQL directly or calls the mock LLM on a miss."""

    def __init__(self, medha_instance: Medha) -> None:
        self._medha = medha_instance
        self.call_count = 0

    @component.output_types(sql=str)
    def run(self, question: str, sql: Optional[str] = None) -> dict:
        if sql:
            return {"sql": sql}
        self.call_count += 1
        result = mock_llm(question)
        _run_async(self._medha.store(question, result))
        return {"sql": result}


# Build the Haystack pipeline
hs_cache = HaystackMedhaCacheComponent(medha)
hs_llm = HaystackMockLLMComponent(medha)

hs_pipeline = Pipeline()
hs_pipeline.add_component("cache", hs_cache)
hs_pipeline.add_component("llm",   hs_llm)

hs_pipeline.connect("cache.question", "llm.question")
hs_pipeline.connect("cache.sql",      "llm.sql")

print("Haystack Pipeline built: cache → llm (short-circuits on hit)")

In [ ]:
test_questions_hs = [
    ("Show the top 10 most expensive products",  "seed — cache hit"),
    ("Top products ranked by cost",               "semantic hit"),
    ("Count orders placed today",                 "seed — cache hit"),
    ("How many orders were submitted today?",     "semantic hit"),
    ("List all tables in the database",           "miss — LLM called"),
]

print("Haystack pipeline — results:")
print(f"  {'Question':<45s}  {'Resolved by':<20s}  Note")
print("  " + "-" * 85)

_hs_hits_before = hs_cache.hit_count

for question, note in test_questions_hs:
    hits_before = hs_cache.hit_count
    result = hs_pipeline.run({"cache": {"question": question}})
    sql = result["llm"]["sql"]
    resolver = "cache" if hs_cache.hit_count > hits_before else "llm"
    print(f"  {question:<45s}  {resolver:<20s}  {note}")

print(f"\nLLM calls made: {hs_llm.call_count} / {len(test_questions_hs)} requests")
print(f"Cache hits: {hs_cache.hit_count}  |  Cache misses: {hs_cache.miss_count}")

---
## Cell 8: Side-by-Side Summary

Aggregate stats across all three frameworks using the same Medha instance.

In [ ]:
total_lc_requests = len(test_questions_lc)
total_li_requests = len(test_questions_li)
total_hs_requests = len(test_questions_hs)

lc_hits = total_lc_requests - _lc_llm_calls
li_hits = total_li_requests - workflow.llm_call_count
hs_hits = total_hs_requests - hs_llm.call_count

print("=" * 60)
print(f"  {'Framework':<15s}  {'Requests':>9s}  {'Cache hits':>10s}  {'Hit rate':>9s}")
print("-" * 60)
for name, reqs, hits in [
    ("LangChain",   total_lc_requests, lc_hits),
    ("LlamaIndex",  total_li_requests, li_hits),
    ("Haystack",    total_hs_requests, hs_hits),
]:
    rate = hits / reqs if reqs else 0
    print(f"  {name:<15s}  {reqs:>9d}  {hits:>10d}  {rate:>8.0%}")
print("=" * 60)

total_reqs = total_lc_requests + total_li_requests + total_hs_requests
total_hits = lc_hits + li_hits + hs_hits
print(f"  {'TOTAL':<15s}  {total_reqs:>9d}  {total_hits:>10d}  {total_hits/total_reqs:>8.0%}")
print()
print(f"LLM calls saved: {total_hits} out of {total_reqs}")

---
## Cell 9: Production Notes

| Topic | Recommendation |
|---|---|
| **Backend** | Replace `InMemoryBackend` with a persistent backend for durability (see table below) |
| **Async in sync frameworks** | LangChain and Haystack are sync-first; wrap async Medha calls with `asyncio.get_event_loop().run_until_complete()` or run inside an async context |
| **LlamaIndex async** | LlamaIndex 0.10+ supports `arun()` — prefer `async_run_component()` for production |
| **Shared instance** | One `Medha` instance across all frameworks is fine — all backends are thread-safe |
| **TTL** | Set `default_ttl_seconds` in `Settings` to auto-expire stale queries after schema changes |
| **Embedder** | `FastEmbedAdapter` runs locally (no API cost). For multilingual, switch to `CohereAdapter` or `GeminiAdapter` (Medha 0.3.1) |

### Choosing a persistent backend

| `backend_type` | Extra deps | Best for |
|---|---|---|
| `"memory"` | none | **Default** in 0.3.1; tests, CI, zero-infra demos |
| `"qdrant"` | `medha-archai[qdrant]` | Vector-first production, HNSW + quantization |
| `"pgvector"` | `medha-archai[pgvector]` | Teams already on PostgreSQL |
| `"elasticsearch"` | `medha-archai[elasticsearch]` | Teams on Elastic stack |
| `"vectorchord"` | `medha-archai[vectorchord]` | High-throughput PostgreSQL workloads |
| `"chroma"` | `medha-archai[chroma]` | Lightweight local vector store |
| `"weaviate"` | `medha-archai[weaviate]` | Managed cloud vector search |

### Switching to a persistent backend

```python
# Drop-in replacement — no other code changes needed
settings = Settings(
    backend_type="qdrant",        # or "pgvector", "elasticsearch", etc.
    qdrant_mode="docker",
    qdrant_host="localhost",
)
medha = Medha("production_cache", embedder=embedder, settings=settings)
await medha.start()
```


In [ ]:
await medha.close()
print("Medha closed. All three framework integrations completed successfully.")